In [1]:

import pandas as pd

df = pd.read_csv("patient_data.csv")

print("Dataset Loaded Successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nDuplicate Patient IDs:", df["patient_id"].duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

display(df.head())


C:\Users\ravir\ Anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ravir\ Anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Dataset Loaded Successfully!
Rows: 100
Columns: 35

Duplicate Patient IDs: 0

Missing Values:
patient_id              0
visit_reason            0
doctor_department       0
appointment_id          0
appointment_datetime    0
appointment_status      0
visit_id                0
visit_datetime          0
prescription_id         0
doctor_id               0
dtype: int64


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,...,visit_datetime,visit_reason,prescription_id,prescription_date,medicine_name_at_visit,medicine_dosage,medicine_frequency,medicine_duration,prescription_instructions,data_type
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,...,2024-04-24 10:30,Type 2 Diabetes,RX0001,2024-04-24,Metformin 500mg,500mg,As recorded by clinician,To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,...,2024-01-14 10:30,Heart Failure,RX0002,2024-01-14,Furosemide 40mg,40mg,As recorded by clinician,To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,...,2024-08-04 10:30,"Type 2 Diabetes, CKD",RX0003,2024-08-04,Insulin Glargine 10U; Losartan 50mg,10U; 50mg,As recorded by clinician; As recorded by clini...,To be confirmed; To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
3,P0004,Anjali Singh,34,Female,"COPD, Heart Failure","Tiotropium 18mcg, Furosemide 40mg",FEV1%,33.8,%,2024-01-23,...,2024-01-23 10:30,"COPD, Heart Failure",RX0004,2024-01-23,Tiotropium 18mcg; Furosemide 40mg,18mcg; 40mg,As recorded by clinician; As recorded by clini...,To be confirmed; To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
4,P0005,Vikram Nair,51,Male,"Hypertension, Heart Failure","Amlodipine 5mg, Furosemide 40mg",BP Systolic,124.9,mmHg,2024-04-26,...,2024-04-26 10:30,"Hypertension, Heart Failure",RX0005,2024-04-26,Amlodipine 5mg; Furosemide 40mg,5mg; 40mg,As recorded by clinician; As recorded by clini...,To be confirmed; To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE


In [2]:

import pandas as pd
import gradio as gr
import html

# Check that the dataset is already loaded
if "df" not in globals():
    raise ValueError("Please run the CSV loading cell first.")

# Check required column
if "patient_id" not in df.columns:
    raise ValueError("The dataset must contain a patient_id column.")

def search_patient(patient_id):
    if not patient_id or not str(patient_id).strip():
        return "<p>Please enter a Patient ID.</p>"

    search_id = str(patient_id).strip()

    # Handle IDs stored as numbers or strings
    ids = df["patient_id"].astype(str).str.strip()
    matches = df[ids == search_id]

    if matches.empty:
        try:
            numeric_id = float(search_id)
            numeric_ids = pd.to_numeric(
                df["patient_id"], errors="coerce"
            )
            matches = df[numeric_ids == numeric_id]
        except ValueError:
            pass

    if matches.empty:
        return f"""
        <div class="record-card">
            <h2>Patient Not Found</h2>
            <p>No record found for Patient ID:
            {html.escape(search_id)}</p>
        </div>
        """

    cards = []

    for _, patient in matches.iterrows():
        details = []

        for column, value in patient.items():
            if pd.isna(value):
                value = "Not Available"

            details.append(f"""
                <div class="detail">
                    <div class="field">
                        {html.escape(str(column).replace("_", " ").title())}
                    </div>
                    <div class="value">
                        {html.escape(str(value))}
                    </div>
                </div>
            """)

        cards.append(f"""
            <div class="record-card">
                <h2>Patient Medical Record</h2>
                <p class="subtitle">Patient ID: {html.escape(search_id)}</p>
                <div class="details-grid">
                    {''.join(details)}
                </div>
            </div>
        """)

    return "".join(cards)


custom_css = """
.gradio-container {
    background: #f3f7fa !important;
}
.record-card {
    background: white;
    border: 1px solid #dce7ee;
    border-radius: 14px;
    padding: 22px;
    margin: 12px 0;
    color: #243447;
    box-shadow: 0 3px 12px rgba(0,0,0,0.05);
}
.record-card h2 {
    color: #087f8c;
    margin-bottom: 6px;
}
.subtitle {
    color: #64748b;
    margin-bottom: 20px;
}
.details-grid {
    display: grid;
    grid-template-columns: repeat(2, minmax(0, 1fr));
    gap: 12px;
}
.detail {
    background: #f5f9fc;
    border-radius: 8px;
    padding: 12px;
    overflow-wrap: anywhere;
}
.field {
    color: #64748b;
    font-size: 12px;
    margin-bottom: 6px;
}
.value {
    font-weight: 600;
    font-size: 14px;
}
@media (max-width: 600px) {
    .details-grid {
        grid-template-columns: 1fr;
    }
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="teal"),
    css=custom_css,
    title="Intelligent Medical Records Assistant"
) as demo:

    gr.Markdown("""
    # Intelligent Medical Records Assistant
    Search a patient and view their complete medical record.
    """)

    with gr.Row():
        patient_input = gr.Textbox(
            label="Patient ID",
            placeholder="Enter Patient ID..."
        )
        search_button = gr.Button(
            "Search Patient",
            variant="primary"
        )

    output = gr.HTML(
        value="<p>Enter a Patient ID to view the medical record.</p>"
    )

    search_button.click(
        fn=search_patient,
        inputs=patient_input,
        outputs=output
    )

    patient_input.submit(
        fn=search_patient,
        inputs=patient_input,
        outputs=output
    )

demo.launch()


C:\Users\ravir\AppData\Local\Temp\ipykernel_22772\1940874107.py:123: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:

# Step 1: Inspect the existing medical records dataset

import pandas as pd

# Confirm that the existing DataFrame is available
if "df" not in globals():
    raise ValueError(
        "DataFrame 'df' not found. Run your CSV loading cell first."
    )

print("DATASET OVERVIEW")
print("-" * 40)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nACTUAL COLUMN NAMES")
for index, column in enumerate(df.columns, start=1):
    print(f"{index}. {column}")

print("\nDATA TYPES")
print(df.dtypes.to_string())

print("\nSAMPLE RECORDS")
display(df.head(3))

print("\nMISSING VALUES")
display(
    df.isnull().sum()
      .sort_values(ascending=False)
      .head(15)
)


DATASET OVERVIEW
----------------------------------------
Rows: 100
Columns: 35

ACTUAL COLUMN NAMES
1. patient_id
2. patient_name
3. age
4. gender
5. diagnosis
6. current_medication
7. lab_test
8. lab_value
9. lab_unit
10. last_visit_date
11. next_scheduled_visit
12. days_since_last_visit
13. missed_last_appointment
14. vitals_bp_systolic
15. vitals_bp_diastolic
16. vitals_heart_rate
17. vitals_spo2
18. notes
19. doctor_id
20. doctor_name
21. doctor_department
22. appointment_id
23. appointment_datetime
24. appointment_status
25. visit_id
26. visit_datetime
27. visit_reason
28. prescription_id
29. prescription_date
30. medicine_name_at_visit
31. medicine_dosage
32. medicine_frequency
33. medicine_duration
34. prescription_instructions
35. data_type

DATA TYPES
patient_id                       str
patient_name                     str
age                            int64
gender                           str
diagnosis                        str
current_medication               str
lab_te

,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,...,visit_datetime,visit_reason,prescription_id,prescription_date,medicine_name_at_visit,medicine_dosage,medicine_frequency,medicine_duration,prescription_instructions,data_type
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,...,2024-04-24 10:30,Type 2 Diabetes,RX0001,2024-04-24,Metformin 500mg,500mg,As recorded by clinician,To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,...,2024-01-14 10:30,Heart Failure,RX0002,2024-01-14,Furosemide 40mg,40mg,As recorded by clinician,To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,...,2024-08-04 10:30,"Type 2 Diabetes, CKD",RX0003,2024-08-04,Insulin Glargine 10U; Losartan 50mg,10U; 50mg,As recorded by clinician; As recorded by clini...,To be confirmed; To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE



MISSING VALUES


patient_id                0
visit_reason              0
doctor_department         0
appointment_id            0
appointment_datetime      0
appointment_status        0
visit_id                  0
visit_datetime            0
prescription_id           0
doctor_id                 0
prescription_date         0
medicine_name_at_visit    0
medicine_dosage           0
medicine_frequency        0
medicine_duration         0
dtype: int64

In [3]:

# Step 2: Retrieve a patient record using Patient ID

import pandas as pd

# Verify the existing dataset
if "df" not in globals():
    raise ValueError("Run the CSV loading cell first.")

# Detect the Patient ID column without assuming exact capitalization
id_candidates = [
    col for col in df.columns
    if str(col).strip().lower().replace(" ", "_") == "patient_id"
]

if not id_candidates:
    print("Available columns:", df.columns.tolist())
    raise ValueError(
        "Patient ID column not found. Check the actual column names."
    )

ID_COLUMN = id_candidates[0]

def get_patient_record(patient_id):
    """Return all rows matching a Patient ID."""

    if patient_id is None or not str(patient_id).strip():
        return None

    search_id = str(patient_id).strip()
    ids = df[ID_COLUMN].astype("string").str.strip()

    matches = df[ids == search_id]

    # Handle IDs stored as numbers in the CSV
    if matches.empty:
        numeric_search = pd.to_numeric(
            pd.Series([search_id]), errors="coerce"
        ).iloc[0]

        if pd.notna(numeric_search):
            numeric_ids = pd.to_numeric(
                df[ID_COLUMN], errors="coerce"
            )
            matches = df[numeric_ids == numeric_search]

    if matches.empty:
        return None

    return matches.copy()


# Test using an ID that actually exists in the dataset
available_ids = df[ID_COLUMN].dropna()

if available_ids.empty:
    print("No valid Patient IDs found in the dataset.")
else:
    test_id = available_ids.iloc[0]
    print("Testing Patient ID:", test_id)

    patient_record = get_patient_record(test_id)

    if patient_record is not None:
        print("Record found!")
        display(patient_record)
    else:
        print("Patient record not found.")


Testing Patient ID: P0001
Record found!


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,...,visit_datetime,visit_reason,prescription_id,prescription_date,medicine_name_at_visit,medicine_dosage,medicine_frequency,medicine_duration,prescription_instructions,data_type
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,...,2024-04-24 10:30,Type 2 Diabetes,RX0001,2024-04-24,Metformin 500mg,500mg,As recorded by clinician,To be confirmed,"SYNTHETIC DEMO ONLY — verify medicine, dose, f...",SYNTHETIC_DEMO_NOT_FOR_CLINICAL_USE


In [4]:

# Step 3: Convert patient records into readable text

def patient_record_to_text(patient_record):

    if patient_record is None or patient_record.empty:
        return "No patient record found."

    lines = []

    for row_number, (_, row) in enumerate(
        patient_record.iterrows(), start=1
    ):
        lines.append(f"Record {row_number}")

        for column, value in row.items():

            if pd.isna(value) or str(value).strip() == "":
                value = "Not available"

            field_name = str(column).replace("_", " ").title()

            lines.append(f"{field_name}: {value}")

        lines.append("")

    return "\n".join(lines)


# Test with the patient_record from Step 2
if patient_record is not None:
    patient_text = patient_record_to_text(patient_record)
    print(patient_text)


Record 1
Patient Id: P0001
Patient Name: Aarav Sharma
Age: 68
Gender: Male
Diagnosis: Type 2 Diabetes
Current Medication: Metformin 500mg
Lab Test: HbA1c
Lab Value: 9.9
Lab Unit: %
Last Visit Date: 2024-04-24
Next Scheduled Visit: 2024-06-08
Days Since Last Visit: 251
Missed Last Appointment: No
Vitals Bp Systolic: 174
Vitals Bp Diastolic: 70
Vitals Heart Rate: 95
Vitals Spo2: 96
Notes: Patient reports good medication adherence.
Doctor Id: D001
Doctor Name: Dr. Ananya Verma
Doctor Department: Endocrinology
Appointment Id: APT0001
Appointment Datetime: 2024-04-24 10:00
Appointment Status: Completed (demo assumption)
Visit Id: VIS0001
Visit Datetime: 2024-04-24 10:30
Visit Reason: Type 2 Diabetes
Prescription Id: RX0001
Prescription Date: 2024-04-24
Medicine Name At Visit: Metformin 500mg
Medicine Dosage: 500mg
Medicine Frequency: As recorded by clinician
Medicine Duration: To be confirmed
Prescription Instructions: SYNTHETIC DEMO ONLY — verify medicine, dose, frequency, duration and ins

In [5]:

# Step 4: Rule-based medical records chatbot

def medical_chatbot(patient_id, question):

    if not question or not question.strip():
        return "Please enter a question."

    record = get_patient_record(patient_id)

    if record is None or record.empty:
        return "Patient not found. Please check the Patient ID."

    question_lower = question.lower()

    # Return the complete record
    if any(word in question_lower for word in [
        "all details", "full record", "complete record",
        "summary", "summarize"
    ]):
        return patient_record_to_text(record)

    # Search the actual column names
    results = []

    for column in record.columns:
        column_name = str(column).lower().replace("_", " ")

        if any(word in column_name for word in question_lower.split()):
            values = record[column].dropna().astype(str).unique()

            if len(values) > 0:
                results.append(
                    f"{column}: {', '.join(values[:10])}"
                )

    if results:
        return "\n".join(results)

    return (
        "I could not match your question to a specific field. "
        "Please ask for the complete record or use a field name "
        "from the dataset."
    )


# Test the chatbot
print(
    medical_chatbot(
        test_id,
        "Show the complete record"
    )
)


Record 1
Patient Id: P0001
Patient Name: Aarav Sharma
Age: 68
Gender: Male
Diagnosis: Type 2 Diabetes
Current Medication: Metformin 500mg
Lab Test: HbA1c
Lab Value: 9.9
Lab Unit: %
Last Visit Date: 2024-04-24
Next Scheduled Visit: 2024-06-08
Days Since Last Visit: 251
Missed Last Appointment: No
Vitals Bp Systolic: 174
Vitals Bp Diastolic: 70
Vitals Heart Rate: 95
Vitals Spo2: 96
Notes: Patient reports good medication adherence.
Doctor Id: D001
Doctor Name: Dr. Ananya Verma
Doctor Department: Endocrinology
Appointment Id: APT0001
Appointment Datetime: 2024-04-24 10:00
Appointment Status: Completed (demo assumption)
Visit Id: VIS0001
Visit Datetime: 2024-04-24 10:30
Visit Reason: Type 2 Diabetes
Prescription Id: RX0001
Prescription Date: 2024-04-24
Medicine Name At Visit: Metformin 500mg
Medicine Dosage: 500mg
Medicine Frequency: As recorded by clinician
Medicine Duration: To be confirmed
Prescription Instructions: SYNTHETIC DEMO ONLY — verify medicine, dose, frequency, duration and ins

In [9]:

import gradio as gr

def respond(patient_id, message, history):
    history = list(history or [])

    if not message or not message.strip():
        answer = "Please enter a question."

    elif not patient_id or not str(patient_id).strip():
        answer = "Please enter a Patient ID first."

    else:
        answer = medical_chatbot(patient_id, message)

    # Use dictionary format for chat history
    history.append({
        "role": "user",
        "content": message
    })

    history.append({
        "role": "assistant",
        "content": answer
    })

    return history, ""


with gr.Blocks(
    title="Intelligent Medical Records Assistant"
) as demo:

    gr.Markdown("""
    # Intelligent Medical Records Assistant
    ### Search patient records without an API
    """)

    patient_id = gr.Textbox(
        label="Patient ID",
        placeholder="Enter Patient ID"
    )

    chatbot = gr.Chatbot(height=400)

    message = gr.Textbox(
        label="Ask a Question",
        placeholder="Show the complete record"
    )

    send = gr.Button("Ask", variant="primary")

    send.click(
        fn=respond,
        inputs=[patient_id, message, chatbot],
        outputs=[chatbot, message]
    )

    message.submit(
        fn=respond,
        inputs=[patient_id, message, chatbot],
        outputs=[chatbot, message]
    )

demo.launch(theme=gr.themes.Soft())


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
